In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

smodel = SentenceTransformer('snunlp/KR-SBERT-V40K-klueNLI-augSTS')
dense_embeddings = smodel.encode(['학교', '공부', '운동'])
cosine_similarity(dense_embeddings) # 코사인 유사도, 단어 간 유사도 계산

In [ ]:
import numpy as np

# one-hot encoding: 단어 간 관계 표현 X (-)
word_dict = {'school': np.array([[1, 0, 0]]), 'study': np.array([[0, 1, 0]]), 'workout': np.array([[0, 0, 1]])}

consine_school_study = cosine_similarity(word_dict['school'], word_dict['study'])
consine_school_workout = cosine_similarity(word_dict['school'], word_dict['workout'])
print(consine_school_study)
print(consine_school_workout)

In [ ]:
from sentence_transformers import SentenceTransformer, models

word_embedding_model = models.Transformer('klue/roberta-base') # 언어 모델 - BERT
pooling_model = models.Pooling(word_embedding_model.get_word_embedding_dimension()) # 풀링 층
model = SentenceTransformer(modules=[word_embedding_model, pooling_model]) # 바이 인코더 생성
model

In [ ]:
# 풀링 모드: 고정된 크기의 문장 임베딩으로 통합. 문장 임베딩 차원 고정.
# 1. 클래스 모드(pooling_mode_cls_tokens): BERT 모델의 첫 번째 임베딩(첫 번째 토큰[CLS]의 출력 임베딩)을 문장 임베딩으로 사용.

# 2. 평균 모드(pooling_mode_mean_tokens)多: BERT 모델의 모든 출력 임베딩의 평균 값을 문장 임베딩으로 사용.
def mean_pooling(model_output, attention_mask): # 언어 모델 출력, 패딩 토큰 위치
    token_embeddings = model_output[0] # 언어 모델 출력 중  마지막 출력만 사용.
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, 1)
    sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)
    return sum_embeddings / sum_mask

In [ ]:
# 3. 최대 모드(pooling_mode_max_tokens): BERT 모델의 모든 출력 임베딩에서 문장 길이(sequence) 최댓값을 문장 임베딩으로 사용.
def max_pooling(model_output, attention_mask):
    token_embeddings = model_output[0]
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    token_embeddings[input_mask_expanded == 0] = -1e9 # 아주 작은 값을 입력해 최댓값이 될 수 없도록 설정.
    return torch.max(token_embeddings, 1)[0]


In [ ]:
from sentence_transformers import SentenceTransformer, util

model = SentenceTransformer('snunlp/KR-SBERT-V40K-klueNLI-augSTS') # 한국어 문장 임베딩 모델
embs = model.encode(['잠이 안 옵니다', # 문장 임베딩으로 변환.
                     '졸음이 옵니다',
                     '기차가 옵니다'])
cos_scores = util.cos_sim(embs, embs) # 문장 임베딩 사이의 코사인 유사도 계산.
print(cos_scores)

In [ ]:
from IPython.utils import text
from PIL import Image

model = SentenceTransformer('clip-ViT-B-32')
# CLIP(Contrastive Language-Image Pre-training) 모델
# ㄴ OpenAI 개발.
# ㄴ 텍스트-이미지 Multi-Modal
# ㄴ 텍스트와 이미지를 동일한 벡터 공간의 임베딩으로 변환하기 때문에 서로 유사도 계산 가능.

img_embs = model.encode([Image.open('dog.jpg'), Image.open('cat.jpg')]) # 이미지 임베딩 변환.
text_embs = model.encode(['A dog on grass', 'Brown cat on yellow background']) # 텍스트 임베딩 변환.

cos_scores = util.cos_sim(img_embs, text_embs) # 코사인 유사도 계산.
print(cos_scores)

In [ ]:
# 의미 검색(semantic search)
from datasets import load_dataset

klue_mrc_dataset = load_dataset('klue', 'mrc', split='train') # 기사 본문과 질문 데이터셋.
sentence_model = SentenceTransformer('snunlp/KR-SBERT-V40K-klueNLI-augSTS') # 한국어 임베딩 모델.

klue_mrc_dataset = klue_mrc_dataset.train_test_split(train_size=1000, shuffle=False)['train']
# 앞에서부터 1,000개의 데이터만 추출.
# 학습, 테스트 데이터 구분.

embeddings = sentence_model.encode(klue_mrc_dataset['context']) # 문장 임베딩 변환.
embeddings.shape

In [ ]:
!pip install faiss-cpu -qqq

In [ ]:
import faiss

index = faiss.IndexFlatIP(embeddings.shape[1]) # KNN(K-Nearest Neighbor) 알고리즘 활용한 검색 인덱스 생성.
index.add(embeddings) # 인덱스에 임베딩 저장.

In [ ]:
# 의미 검색 장점: 키워드가 동일하지 않아도 의미가 유사하면 찾을 수 있다.
query = "이번 연도에는 언제 비가 많이 올까?"
query_embedding = sentence_model.encode([query]) # 문장 임베딩 변환.
distances, indices = index.search(query_embedding, 3) # 검색

for idx in indices[0]:
    idx = int(idx)
    print(klue_mrc_dataset['context'][idx][:50])

In [ ]:
# 의미 검색 단점: 관련성 없는 검색 결과가 나오기도 함.
query = klue_mrc_dataset[3]['question']
query_embedding = sentence_model.encode([query])
distances, indices = index.search(query_embedding, 3)

for idx in indices[0]:
    idx = int(idx)
    print(klue_mrc_dataset['context'][idx][:50])

In [ ]:
# BM25 구현: 키워드 검색
import math
import numpy as np
from typing import List
from transformers import PreTrainedTokenizer
from collections import defaultdict

class BM25:
    def __init__(self, corpus: List[List[str]], tokenizer: PreTrainedTokenizer):
        self.tokenizer = tokenizer
        self.corpus = corpus
        self.tokenized_corpus = self.tokenizer(corpus, add_sepcial_tokens=False)['input_ids']
        self.n_docs = len(self.tokenized_corpus)
        self.avg_doc_lens = sum(len(lst) for lst in self.tokenized_corpus) / len(self.tokenized_corpus)
        self.idf = self._calculate_idf()
        self.term_freqs = self._calculate_term_freqs()

    def _calculate_idf(self):
        idf = defaultdict(float)
        for doc in self.tokenized_corpus:
            for token in set(doc):
                idf[token] += 1
        for token_id, doc_frequency in idf.items():
            idf[token_id] = math.log(((self.n_docs - doc_frequency + 0.5) / (doc_frequency + 0.5)) + 1)
        return idf

    def _calculate_term_freqs(self):
        term_freqs = [defaultdict(int) for _ in range(self.n_docs)]
        for i, doc in enumerate(self.tokenized_corpus):
            for token_id in doc:
                term_freqs[i][token_id] += 1
        return term_freqs

    def get_scores(self, query:str, k1:float=1.2, b:float=0.75):
        query = self.tokenizer([query], add_special_tokens=False)['input_ids'][0]
        scores = np.zeros(self.n_docs)
        for q in query:
            idf = self.idf[q]
            for i, term_freq in enumerate(self.term_freqs):
                q_freqency = term_freq[q]
                doc_len = len(self.tokenized_corpus[i])
                score_q = idf * (q_freqency * (k1 + 1)) / ((q_freqency) + k1 * (1 - b + b * (doc_len / self.avg_doc_lens)))
                scores[i] += score_q
        return scores

    def get_top_k(self, query:str, k:int):
        scores = self.get_scores(query)
        top_k_indices = np.argsort(scores)[-k:][::-1]
        top_k_scores = scores[top_k_indices]
        return top_k_scores, top_k_indices

In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained('klue/roberta-base')

bm25 = BM25(['안녕하세요', '반갑습니다', '안녕 서울'], tokenizer)
bm25.get_scores('안녕')

In [ ]:
bm25 = BM25(list(klue_mrc_dataset['context']), tokenizer)

query = klue_mrc_dataset[3]['question']
print('query:', query)
_, bm25_search_ranking = bm25.get_top_k(query, 100)

for idx in bm25_search_ranking[:3]:
    idx = int(idx)
    print(klue_mrc_dataset['context'][idx][:50])

In [ ]:
# 상호 순위 조합(RRF): = 통계 기반 점수(BM25) + 임베딩 유사도 점수(의미 검색)
from collections import defaultdict

def reciprocal_rank_fusion(rankings:List[List[int]], k=5):
    rrf = defaultdict(float)
    for ranking in rankings:
        for i, doc_id in enumerate(ranking, 1):
            rrf[doc_id] += 1.0 / (k + i)
    return sorted(rrf.items(), key=lambda x: x[1], reverse=True)


In [ ]:
rankings = [[1,4,3,5,6], [2,1,3,6,4]]
reciprocal_rank_fusion(rankings)

In [ ]:
# 하이브리드 검색
def dense_vector_search(query:str, k:int):
    query_embedding = sentence_model.encode([query])
    distances, indices = index.search(query_embedding, k)
    return distances[0], indices[0]

def hybrid_search(query, k=20):
    _, dense_search_ranking = dense_vector_search(query, 100)
    _, bm25_search_ranking = bm25.get_top_k(query, 100)

    results = reciprocal_rank_fusion([dense_search_ranking, bm25_search_ranking], k=k)
    return results

In [ ]:
query = "이번 연도에는 언제 비가 많이 올까?"
print("검색 쿼리 문장:", query)
results = hybrid_search(query)
for idx, score in results[:3]:
    idx = int(idx)
    print(klue_mrc_dataset['context'][idx][:50])

print('='*80)

query = klue_mrc_dataset[3]['question']
print("검색 쿼리 문장:", query)
results = hybrid_search(query)
for idx, score in results[:3]:
    idx = int(idx)
    print(klue_mrc_dataset['context'][idx][:50])